In [0]:
%sql
CREATE OR REPLACE TABLE mlo.features.automl_v3 AS
SELECT f.*, l.Bad
FROM mlo.features.weather_daily_v3 VERSION AS OF 0 AS f
JOIN mlo.features.weather_labels AS l
  ON f.date = l.date AND f.station = l.station

In [0]:
%pip install flaml[automl]

In [0]:
import mlflow
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pdf = spark.table("mlo.features.automl_v3").toPandas().sort_values("date").reset_index(drop=True)

FEATURES = [c for c in pdf.columns if c not in ("station", "date", "Bad")]
split = int(len(pdf) * 0.8)                     # chronological, matching 03
train, test = pdf.iloc[:split], pdf.iloc[split:]
Xtr, ytr = train[FEATURES], train["Bad"]
Xte, yte = test[FEATURES], test["Bad"]

print(f"{len(FEATURES)} features | "
      f"train {len(train)} ({train.date.min().date()} -> {train.date.max().date()}) | "
      f"test {len(test)} ({test.date.min().date()} -> {test.date.max().date()})")

user = spark.sql("select current_user()").first()[0]
mlflow.set_experiment(f"/Users/{user}/weather_experiments")   # same home as the v1/v2 runs

CANDIDATES = {
    "logreg": Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced",
                                   solver="liblinear", random_state=42))]),
    # L1 does the feature selection that 44 features against ~475 events demands.
    "logreg_l1": Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", penalty="l1",
                                   C=0.1, solver="liblinear", random_state=42))]),
    "random_forest": Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(n_estimators=400, class_weight="balanced",
                                       min_samples_leaf=5, random_state=42, n_jobs=-1))]),
    # Handles NaN natively — no imputer, which is the honest treatment for a table
    # where a null means "that station didn't report", not "unknown value".
    "hist_gbm": HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05,
                                               random_state=42),
}

results = []
for name, model in CANDIDATES.items():
    with mlflow.start_run(run_name=f"{name}_v3"):
        mlflow.set_tags({"feature_set_version": "v3", "model_type": name,
                         "target": "Bad_t+1", "source": "manual_sweep"})
        mlflow.log_param("n_features", len(FEATURES))
        model.fit(Xtr, ytr)
        pred, proba = model.predict(Xte), model.predict_proba(Xte)[:, 1]
        m = {"f1": f1_score(yte, pred, zero_division=0),
             "roc_auc": roc_auc_score(yte, proba),
             "accuracy": accuracy_score(yte, pred),
             "precision": precision_score(yte, pred, zero_division=0),
             "recall": recall_score(yte, pred, zero_division=0)}
        mlflow.log_metrics(m)
        mlflow.sklearn.log_model(model, name)
        results.append({"model": name, **m})

print(pd.DataFrame(results).sort_values("f1", ascending=False).to_string(index=False))

In [0]:
from flaml import AutoML

# FLAML's linear learners choke on NaN. Impute with TRAIN medians for both splits —
# using test medians would leak.
med = Xtr.median()
Xtr_f, Xte_f = Xtr.fillna(med), Xte.fillna(med)

fa = AutoML()
with mlflow.start_run(run_name="flaml_v3"):
    mlflow.set_tags({"feature_set_version": "v3", "model_type": "flaml",
                     "target": "Bad_t+1", "source": "flaml"})
    fa.fit(X_train=Xtr_f, y_train=ytr, task="classification", metric="f1",
           time_budget=600, mlflow_logging=False, verbose=1)

    pred, proba = fa.predict(Xte_f), fa.predict_proba(Xte_f)[:, 1]
    mlflow.log_params({"best_estimator": fa.best_estimator, "n_features": len(FEATURES)})
    mlflow.log_metrics({"f1": f1_score(yte, pred, zero_division=0),
                        "roc_auc": roc_auc_score(yte, proba),
                        "accuracy": accuracy_score(yte, pred),
                        "precision": precision_score(yte, pred, zero_division=0),
                        "recall": recall_score(yte, pred, zero_division=0)})
    mlflow.sklearn.log_model(fa.model.estimator, "flaml_best")

print(fa.best_estimator, fa.best_config)